# ReviewRing AI — Colab training

Clone the repository or unpack the supplied updated source bundle, then run the cells in order.
The trainers use CPU; MiniLM text embedding can use an available GPU. A GPU is optional.
Each experiment gets a separate folder under `experiments/`, containing resolved configurations,
processed data, model artifacts, and `RESULTS.md`. Old committed results are never included.

The full experiment compares all seven Amazon model choices on seeds 17, 42 and 73.
Set `SMOKE = True` in the training cell for a two-epoch runtime check first.
Optional upload, separate-track and public UI cells are disabled by default for Run all.


In [ ]:
# Step 0: use an existing checkout/source bundle, or clone once.
REPO_URL = 'https://github.com/kishoreafk/ReviewRing-AI.git'
REPO_DIR = '/content/ReviewRing-AI'
BRANCH = 'main'

import os, subprocess, sys
from pathlib import Path

if not Path(REPO_DIR, 'pyproject.toml').exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    print('Using existing source files. No automatic pull or overwrite.')
os.chdir(REPO_DIR)
if not Path('scripts/run_experiment.py').exists():
    raise RuntimeError('This checkout needs the Colab fixes. Upload/unpack the updated source bundle, or update to the branch containing the fixes.')
print('Python:', sys.version.split()[0], '| Project:', os.getcwd())


In [ ]:
# Step 1: install dependencies in the notebook's Python environment.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-c', 'import torch, reviewring; print("Torch", torch.__version__, "CUDA available:", torch.cuda.is_available(), "ReviewRing", reviewring.__version__)'], check=True)


In [ ]:
# Step 2 (optional): Kaggle credentials. Canonical downloads work without this.
UPLOAD_KAGGLE = False
if UPLOAD_KAGGLE:
    from google.colab import files
    uploaded = files.upload()
    if 'kaggle.json' in uploaded:
        credential = Path.home() / '.kaggle' / 'kaggle.json'
        credential.parent.mkdir(exist_ok=True)
        credential.write_bytes(uploaded['kaggle.json'])
        credential.chmod(0o600)


In [ ]:
# Step 3: regression tests; failure stops this cell before training.
subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q'], check=True)


In [ ]:
# Step 4: download/validate raw datasets.
subprocess.run([sys.executable, 'scripts/get_data.py'], check=True)


## Path A — full isolated experiment

Run all models and all three seeds. Runtime depends on Colab CPU allocation; the
T4 speeds up text embedding only. Failures propagate and the log is saved.


In [ ]:
SMOKE = False  # True: seed 42, two epochs, Yelp graph + Amazon adaptive fusion
# To explore longer Yelp training, edit configs/yelp_static.yaml before this cell.
# Early stopping still uses validation AP. Do not select settings on test metrics.
import shlex
cmd = [sys.executable, '-u', 'scripts/run_experiment.py']
if SMOKE:
    cmd.append('--smoke')
pipeline = shlex.join(cmd) + ' 2>&1 | tee /content/train.log'
subprocess.run(['bash', '-o', 'pipefail', '-c', pipeline], check=True)


In [ ]:
# The current experiment includes its own results and resolved configurations.
import json
experiment = Path(json.loads(Path('latest_experiment.json').read_text())['experiment_dir'])
print('Experiment:', experiment)
print((experiment / 'experiment.json').read_text())
if (experiment / 'RESULTS.md').exists():
    print((experiment / 'RESULTS.md').read_text())


## Path B — optional separate tracks

These cells are disabled by default to avoid running training twice with Run all.
Enable only if you want a separate experiment for each track. For individual
stage commands, use that experiment's saved YAML under `configs/` so reports
continue to use the matching processed data and artifacts.


In [ ]:
RUN_SEPARATE_TRACKS = False
if RUN_SEPARATE_TRACKS:
    subprocess.run([sys.executable, 'scripts/run_experiment.py', '--track', 'yelp'], check=True)


In [ ]:
# Each invocation creates a new directory and preserves previous outputs.
# --epochs N overrides the expert and fusion epoch ceilings.
# --prepared-data copies existing data/processed inputs instead of rebuilding.


In [ ]:
if RUN_SEPARATE_TRACKS:
    subprocess.run([sys.executable, 'scripts/run_experiment.py', '--track', 'amazon'], check=True)


In [ ]:
# Rebuild the table for exactly the latest experiment, without retraining.
experiment = Path(json.loads(Path('latest_experiment.json').read_text())['experiment_dir'])
subprocess.run([sys.executable, 'scripts/collect_results.py',
                '--artifacts-dir', str(experiment / 'artifacts'),
                '--output', str(experiment / 'RESULTS.md')], check=True)


In [ ]:
# Download only this experiment (including processed data needed for evaluation).
import shutil
from google.colab import files
archive = shutil.make_archive('/content/' + experiment.name, 'zip',
                              root_dir=experiment.parent, base_dir=experiment.name)
files.download(archive)
# For persistence, optionally copy the same experiment directory to mounted Drive.


In [ ]:
# Optional investigation UI; intentionally not launched by Run all.
# The existing UI reads artifacts/ by default; isolated experiment runs are
# inspected through their saved reports and RESULTS.md.
# To launch the legacy UI deliberately:
# !streamlit run app/streamlit_app.py --server.port 8711


## Notes

- Every run saves checkpoints, calibrated predictions, configuration and provenance.
- The experiment manifest records completion or failure and the runs already finished.
- Rerunning starts a fresh experiment. Save its directory before Colab disconnects.
- The collector retains one latest run per model and seed within the chosen directory;
  paired comparisons use each common seed once and report sample standard deviation.
- Review-level replay detection and reconstruction of a whole ring are different metrics.
- Track A uses platform-filter proxy labels; Track B evaluates controlled synthetic campaigns.
- A smoke run checks runtime behavior, not predictive quality. Compare full experiments
  under the same data, encoder and settings before drawing performance conclusions.
